In [1]:
from pyspark.sql import SparkSession

# Creating Spark Session
spark = (SparkSession.builder
    .appName("BigData_Lab02.2_Ubuntu") # Spark session name
    .master("local[*]") \
    .config("spark.driver.bindAddress", "127.0.0.1")\
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.ui.port", "4041")
    .config("spark.port.maxRetries", "100")
    .config("spark.driver.memory", "2g")
    .config("spark.jars.packages", 
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1,"
            "org.apache.kafka:kafka-clients:3.6.0,"
            "org.apache.spark:spark-streaming-kafka-0-10_2.13:4.0.1") 
    .getOrCreate())

# Checking success
print(f"Successfully! Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/27 22:07:03 WARN Utils: Your hostname, Kiennguyen, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/27 22:07:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/kiennguyen/bigdata/lab01/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/kiennguyen/.ivy2.5.2/cache
The jars for the packages stored in: /home/kiennguyen/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.kafka#kafka-clients added as a dependency
org.apache.spark#spark-streaming-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dc12e912-794e-4b2d-99ad-6ddb63d47518;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.1 in central

Successfully! Spark version: 4.0.1


In [2]:
import kagglehub

# Download dataset from kaggle
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

# reading data to spark frame
df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

/home/kiennguyen/bigdata/lab01/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from pyspark.sql.types import *

rating_schema = StructType([
    StructField("userId", IntegerType()),
    StructField("movieId", IntegerType()),
    StructField("rating", FloatType()),
    StructField("timestamp", LongType())
])

movie_schema = StructType([
    StructField("movieId", IntegerType()),
    StructField("title", StringType()),
    StructField("genres", StringType())
])

tag_schema = StructType([
    StructField("userId", IntegerType()),
    StructField("movieId", IntegerType()),
    StructField("tag", StringType()),
    StructField("timestamp", LongType())
])

In [ ]:
from pyspark.sql.functions import *
brokers = "localhost:9092,localhost:9192,localhost:9292"
def read_kafka_topic(topic_name, schema):
    return spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", brokers) \
        .option("subscribe", topic_name) \
        .option("startingOffsets", "earliest") \
        .option("maxOffsetsPerTrigger", 100) \
        .load() \
        .selectExpr("CAST(value AS STRING)") \
        .select(from_json(col("value"), schema).alias("data")) \
        .select("data.*") \
        .withColumn("timestamp", to_timestamp(from_unixtime(col("timestamp"))))

ratings_stream = read_kafka_topic("Lab1_ratings", rating_schema)
tags_stream = read_kafka_topic("Lab1_tags", tag_schema)

movies_static = df_movies

In [10]:
hot_genres = ratings_stream.join(movies_static, "movieId") \
    .withColumn("genre", explode(split(col("genres"), "\|"))) \
    .groupBy("genre") \
    .count() \
    .orderBy(col("count").desc())

# 2. Khởi chạy luồng stream [cite: 185, 186]
query2 = hot_genres.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime='5 seconds') \
    .start() # [cite: 187, 188, 189]

print(">>> Start Exercise 2 in 60 second...")
query2.awaitTermination(60) 

# 4. Dừng luồng sau khi hết thời gian timeout
query2.stop()
print(">>> Stop query 2.")

<>:2: SyntaxWarning: invalid escape sequence '\|'
<>:2: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_67344/2834462790.py:2: SyntaxWarning: invalid escape sequence '\|'
  .withColumn("genre", explode(split(col("genres"), "\|"))) \
26/04/27 22:23:25 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-3ac32d96-9ed4-4987-9b86-981335f68252. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/27 22:23:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


>>> Start Exercise 2 in 60 second...


26/04/27 22:23:44 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 18400 milliseconds


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+-----+
|genre    |count|
+---------+-----+
|Action   |37   |
|Comedy   |36   |
|Adventure|33   |
|Drama    |28   |
|Thriller |23   |
|Crime    |18   |
|Romance  |18   |
|Fantasy  |17   |
|Children |17   |
|Musical  |15   |
|Sci-Fi   |14   |
|War      |12   |
|Animation|11   |
|Mystery  |6    |
|Horror   |6    |
|Western  |4    |
+---------+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+---------+-----+
|genre    |count|
+---------+-----+
|Action   |77   |
|Adventure|73   |
|Comedy   |69   |
|Drama    |57   |
|Thriller |47   |
|Crime    |41   |
|Fantasy  |41   |
|Children |39   |
|Sci-Fi   |35   |
|Animation|28   |
|Romance  |25   |
|Musical  |22   |
|War      |17   |
|Horror   |16   |
|Mystery  |15   |
|Western  |6    |
|Film-Noir|1    |
+---------+-----+



...

-------------------------------------------
Batch: 30
-------------------------------------------
+-----------+-----+
|genre      |count|
+-----------+-----+
|Drama      |1337 |
|Comedy     |1125 |
|Action     |923  |
|Adventure  |783  |
|Thriller   |763  |
|Romance    |594  |
|Crime      |549  |
|Sci-Fi     |473  |
|Fantasy    |379  |
|Children   |343  |
|Mystery    |240  |
|Animation  |240  |
|War        |190  |
|Musical    |169  |
|Horror     |163  |
|IMAX       |96   |
|Western    |94   |
|Film-Noir  |32   |
|Documentary|19   |
+-----------+-----+



26/04/27 22:27:30 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 6265 milliseconds


-------------------------------------------
Batch: 31
-------------------------------------------
+-----------+-----+
|genre      |count|
+-----------+-----+
|Drama      |1398 |
|Comedy     |1176 |
|Action     |937  |
|Adventure  |795  |
|Thriller   |783  |
|Romance    |617  |
|Crime      |560  |
|Sci-Fi     |479  |
|Fantasy    |384  |
|Children   |345  |
|Mystery    |254  |
|Animation  |242  |
|War        |192  |
|Musical    |175  |
|Horror     |164  |
|Western    |97   |
|IMAX       |96   |
|Film-Noir  |34   |
|Documentary|19   |
+-----------+-----+



-------------------------------------------
Batch: 32
-------------------------------------------
+-----------+-----+
|genre      |count|
+-----------+-----+
|Drama      |1456 |
|Comedy     |1207 |
|Action     |952  |
|Adventure  |811  |
|Thriller   |806  |
|Romance    |639  |
|Crime      |580  |
|Sci-Fi     |485  |
|Fantasy    |395  |
|Children   |363  |
|Mystery    |260  |
|Animation  |249  |
|War        |198  |
|Musical    |181  |
|Horror     |167  |
|IMAX       |99   |
|Western    |99   |
|Film-Noir  |34   |
|Documentary|19   |
+-----------+-----+



26/04/27 22:27:37 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 7000 milliseconds


In [11]:
# Lấy danh sách tất cả các query đang hoạt động trong SparkSession
active_queries = spark.streams.active

if len(active_queries) > 0:
    print(f"Đang tìm thấy {len(active_queries)} query đang chạy. Tiến hành dừng...")
    for q in active_queries:
        print(f"Đã dừng query: {q.name if q.name else q.id}")
        q.stop()
else:
    print("Không có query nào đang chạy ngầm.")

Đang tìm thấy 1 query đang chạy. Tiến hành dừng...
Đã dừng query: 5f86cf22-0c65-46cd-a2b8-11edc191ac50


26/04/27 22:28:11 WARN DAGScheduler: Failed to cancel job group b80024e1-6524-4e86-8dbd-ec443118bfad. Cannot find active jobs for it.
26/04/27 22:28:12 WARN DAGScheduler: Failed to cancel job group b80024e1-6524-4e86-8dbd-ec443118bfad. Cannot find active jobs for it.


## EX3

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# 1. Tính toán số lượt rate theo cửa sổ 5 phút (Tumbling Window)
# Phải đặt watermark TRƯỚC khi thực hiện groupBy window 
windowed_counts = ratings_stream \
    .withWatermark("timestamp", "10 minutes") \
    .groupBy(
        window(col("timestamp"), "5 minutes"), # Cửa sổ 5 phút 
        col("movieId")
    ) \
    .count() \
    .withColumnRenamed("count", "cnt")

# 2. Join với bảng movies tĩnh để lấy tiêu đề phim [cite: 208, 209]
trending_df = windowed_counts.join(movies_static, "movieId")

# 3. Hàm xử lý xếp hạng cho từng Micro-Batch
def process_trending_batch(batch_df, batch_id):
    # Định nghĩa quy tắc xếp hạng:
    # Phân vùng theo window, sắp xếp theo lượt rate giảm dần,movieId tăng dần [cite: 209, 213]
    window_spec = Window.partitionBy("window").orderBy(col("cnt").desc(), col("movieId").asc())
    
    # Thực hiện xếp hạng và lọc Top 3 
    final_result = batch_df.withColumn("rank", dense_rank().over(window_spec)) \
        .filter(col("rank") <= 3) \
        .select("window", "movieId", "title", "cnt", "rank") \
        .orderBy("window", "rank")
    
    if not final_result.isEmpty():
        print(f"\n--- Batch ID: {batch_id} | Trending Now (Top 3) ---")
        final_result.show(truncate=False)

# 4. Khởi chạy stream với trigger 5 giây [cite: 204]
query3 = trending_df.writeStream \
    .foreachBatch(process_trending_batch) \
    .trigger(processingTime='5 seconds') \
    .start()

# Chạy demo trong 20 giây rồi dừng
print(">>> Đang chạy Exercise 3 trong 60 giây...")
query3.awaitTermination()
# query3.stop()
# print(">>> Demo Exercise 3 kết thúc.")

26/04/27 22:28:19 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-cf81504a-5f5b-450d-b2b3-2616ccd28206. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/27 22:28:19 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


>>> Đang chạy Exercise 3 trong 60 giây...


26/04/27 22:28:29 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 9216 milliseconds



--- Batch ID: 1 | Trending Now (Top 3) ---


+------------------------------------------+-------+-----------------------------------------------------+---+----+
|window                                    |movieId|title                                                |cnt|rank|
+------------------------------------------+-------+-----------------------------------------------------+---+----+
|{2000-07-31 01:05:00, 2000-07-31 01:10:00}|804    |She's the One (1996)                                 |1  |1   |
|{2000-07-31 01:05:00, 2000-07-31 01:10:00}|1210   |Star Wars: Episode VI - Return of the Jedi (1983)    |1  |2   |
|{2000-07-31 01:05:00, 2000-07-31 01:10:00}|2018   |Bambi (1942)                                         |1  |3   |
|{2000-07-31 01:10:00, 2000-07-31 01:15:00}|101    |Bottle Rocket (1996)                                 |1  |1   |
|{2000-07-31 01:10:00, 2000-07-31 01:15:00}|441    |Dazed and Confused (1993)                            |1  |2   |
|{2000-07-31 01:10:00, 2000-07-31 01:15:00}|1473   |Best Men (1997)     

...

26/04/27 22:31:02 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 5287 milliseconds



--- Batch ID: 27 | Trending Now (Top 3) ---


26/04/27 22:31:11 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 8477 milliseconds


+------------------------------------------+-------+--------------------------------------+---+----+
|window                                    |movieId|title                                 |cnt|rank|
+------------------------------------------+-------+--------------------------------------+---+----+
|{2018-02-04 02:25:00, 2018-02-04 02:30:00}|1193   |One Flew Over the Cuckoo's Nest (1975)|1  |1   |
|{2018-03-18 04:50:00, 2018-03-18 04:55:00}|53129  |Mr. Brooks (2007)                     |1  |1   |
|{2018-03-18 07:10:00, 2018-03-18 07:15:00}|51357  |Citizen X (1995)                      |1  |1   |
|{2018-05-13 04:15:00, 2018-05-13 04:20:00}|27878  |Born into Brothels (2004)             |1  |1   |
+------------------------------------------+-------+--------------------------------------+---+----+



26/04/27 22:31:35 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 5264 milliseconds
26/04/27 22:32:05 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 5574 milliseconds
26/04/27 22:32:11 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 5923 milliseconds


## EX4

In [8]:
from pyspark.sql.functions import *

# 1. Thiết lập Watermark cho cả hai luồng (Yêu cầu bắt buộc) 
ratings_with_wm = ratings_stream.withWatermark("timestamp", "10 minutes")
tags_with_wm = tags_stream.withWatermark("timestamp", "10 minutes")

# 2. Thực hiện Stream-Stream Join trên movieId [cite: 220, 225]
# Điều kiện: movieId trùng nhau VÀ thời gian tag không cách quá 5 phút so với rating
joined_stream = ratings_with_wm.alias("r").join(
    tags_with_wm.alias("t"),
    expr("""
        r.movieId = t.movieId AND
        t.timestamp >= r.timestamp - interval 5 minutes AND
        t.timestamp <= r.timestamp + interval 5 minutes
    """),
    "inner"
)

# 3. Chọn các cột đầu ra (movieId, tag, rating, ratingTime, tagTime) [cite: 221]
result_ex4 = joined_stream.select(
    col("r.movieId"),
    col("t.tag"),
    col("r.rating"),
    col("r.timestamp").alias("ratingTime"),
    col("t.timestamp").alias("tagTime")
)

# 4. Ghi kết quả ra console ở chế độ APPEND mỗi 5 giây [cite: 216, 217, 221]
query4 = result_ex4.writeStream \
    .outputMode("append") \
    .format("console") \
    .trigger(processingTime='5 seconds') \
    .start()

print(">>> Đang chạy Exercise 4 (Stream-Stream Join) trong 60 giây...")
query4.awaitTermination()
# query4.stop()
# print(">>> Hoàn thành toàn bộ Lab 02.")

26/04/27 22:10:59 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-d4a9ccb3-16ad-4455-b223-434ae068b789. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/27 22:10:59 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


>>> Đang chạy Exercise 4 (Stream-Stream Join) trong 60 giây...


-------------------------------------------
Batch: 0
-------------------------------------------
+-------+---+------+----------+-------+
|movieId|tag|rating|ratingTime|tagTime|
+-------+---+------+----------+-------+
+-------+---+------+----------+-------+



26/04/27 22:11:31 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 31770 milliseconds
26/04/27 22:11:51 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 20005 milliseconds


-------------------------------------------
Batch: 1
-------------------------------------------
+-------+---+------+----------+-------+
|movieId|tag|rating|ratingTime|tagTime|
+-------+---+------+----------+-------+
+-------+---+------+----------+-------+



...

26/04/27 22:17:12 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 13366 milliseconds


-------------------------------------------
Batch: 25
-------------------------------------------
+-------+---+------+----------+-------+
|movieId|tag|rating|ratingTime|tagTime|
+-------+---+------+----------+-------+
+-------+---+------+----------+-------+



-------------------------------------------
Batch: 26
-------------------------------------------
+-------+---+------+----------+-------+
|movieId|tag|rating|ratingTime|tagTime|
+-------+---+------+----------+-------+
+-------+---+------+----------+-------+



26/04/27 22:17:28 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 16031 milliseconds
26/04/27 22:17:45 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 16578 milliseconds


-------------------------------------------
Batch: 27
-------------------------------------------
+-------+---+------+----------+-------+
|movieId|tag|rating|ratingTime|tagTime|
+-------+---+------+----------+-------+
+-------+---+------+----------+-------+

